In [1]:
!pip install mlflow

!pip install --upgrade mlflow cachetools dagshub transformers -q

!pip install "protobuf<4.0.0" --upgrade
!pip install sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.2.19
    Uninstalling SQLAlchemy-1.2.19:
      Successfully uninstalled SQLAlchemy-1.2.19
ERROR: pip's dependency r

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizer
from tqdm import tqdm
import mlflow
import dagshub

2026-01-13 13:40:08.928427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768311609.119412      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768311609.171846      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768311609.619774      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768311609.619815      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768311609.619817      55 computation_placer.cc:177] computation placer alr

In [ ]:

# --- 1. CONFIGURAZIONE DAGSHUB & MLFLOW ---
DAGSHUB_USER = ""
DAGSHUB_REPO = "AISE_labeling"
DAGSHUB_TOKEN = ""

os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USER
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USER}/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("")
EXPERIMENT_NAME = ''

<Experiment: artifact_location='mlflow-artifacts:/83d4f3c72b0740b5a60088886aa9967e', creation_time=1768242552437, experiment_id='7', last_update_time=1768242552437, lifecycle_stage='active', name='Social_Risk_Multilingual_5', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [6]:
# --- 2. COSTANTI MODELLO ---
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN_MSG = 512
MAX_MSG_PER_CONV = 120
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MSG_LABEL_COLS = ['OVERRELIANCE_DECISION', 'AUTONOMY_LOW_SELF', 'EMOTIONAL_DEPENDENCE', 'PARASOCIAL_REL', 
                  'ANTHROPOMORPHISM_STRONG', 'ADDICTIVE_USE', 'SOCIAL_WITHDRAWAL', 'SKILL_ATROPHY', 
                  'ANXIETY_ESCALATION', 'CRISIS_RISK']
CONV_LABEL_COLS = [c + '_C' for c in MSG_LABEL_COLS]

dtype_dict_1 = {col: 'Int64' for col in MSG_LABEL_COLS}
dtype_dict_2 = {col: 'Int64' for col in CONV_LABEL_COLS}
dtype_dict_2['global_risk_flag'] = 'Int64'


In [7]:
# --- 3. ARCHITETTURA (Identica al training) ---
class HierarchicalModel(nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.m_heads = nn.ModuleList([nn.Linear(h, 3) for _ in range(10)]) 
        self.proj = nn.Linear(h + 30, 256)
        self.gru = nn.GRU(256, 256, num_layers=1, batch_first=True, bidirectional=True)
        self.c_multi_heads = nn.ModuleList([nn.Linear(512, 3) for _ in range(10)])
        self.c_bin_head = nn.Linear(512, 2) 

    def forward(self, ids, mask, lengths):
        B, S, L = ids.shape
        with torch.no_grad():
            emb = self.encoder(ids.view(-1, L), attention_mask=mask.view(-1, L)).last_hidden_state[:, 0, :]
        m_logits = torch.stack([head(emb) for head in self.m_heads], dim=1) 
        combined = self.proj(torch.cat([emb.view(B, S, -1), m_logits.view(B, S, -1)], dim=-1))
        gru_out, _ = self.gru(combined)
        final_state = gru_out[torch.arange(B), (lengths - 1).clamp(min=0)]
        return m_logits.view(B, S, 10, 3), torch.stack([head(final_state) for head in self.c_multi_heads], dim=1), self.c_bin_head(final_state)


In [9]:
# --- 4. FUNZIONE DOWNLOAD MODELLO ---
def download_best_model():
    print(f">>> Ricerca dell'ultimo modello in corso nell'esperimento: {EXPERIMENT_NAME}...")
    
    # Trova l'esperimento
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        raise ValueError(f"Esperimento '{EXPERIMENT_NAME}' non trovato su DagsHub!")

    # Prendi l'ultima run completata con successo
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], 
                             filter_string="status = 'FINISHED'", 
                             order_by=["start_time DESC"], max_results=1)
    
    if runs.empty:
        raise ValueError("Nessuna run terminata trovata!")

    run_id = runs.iloc[0].run_id
    print(f">>> Download artefatto dalla Run ID: {run_id}")

    # Scarica il file specifico dal path definito nel training
    # Nel training usavi: mlflow.log_artifact('best_model.pt', artifact_path="modelli_salvati")
    local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="modelli_salvati/best_model.pt")
    
    print(f">>> Modello scaricato correttamente in: {local_path}")
    return local_path



In [10]:
# --- 5. DATASET PER INFERENZA ---
class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.tokenizer = tokenizer
        self.samples = []
        # Ordiniamo per assicurarci che la sequenza temporale sia corretta
        grouped = df.sort_values(['conversation_id', 'message_id']).groupby('conversation_id')
        
        for conv_id, group in tqdm(grouped, desc="Preparazione dati"):
            texts = group['text'].astype(str).tolist()[:MAX_MSG_PER_CONV]
            roles = group['user_role'].tolist()[:MAX_MSG_PER_CONV]
            indices = group.index.tolist()[:MAX_MSG_PER_CONV]
            
            self.samples.append({
                'conv_id': conv_id, 'texts': texts, 'roles': roles, 
                'indices': indices, 'len': len(texts)
            })

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = self.tokenizer(s['texts'], padding='max_length', truncation=True, max_length=MAX_LEN_MSG, return_tensors='pt')
        pad = MAX_MSG_PER_CONV - s['len']
        ids = torch.cat([enc['input_ids'], torch.zeros((pad, MAX_LEN_MSG), dtype=torch.long)])
        mask = torch.cat([enc['attention_mask'], torch.zeros((pad, MAX_LEN_MSG), dtype=torch.long)])
        return {'ids': ids, 'mask': mask, 'actual_len': s['len'], 'indices': s['indices'], 'roles': s['roles'], 'conv_id': s['conv_id']}



In [11]:
# --- 6. PIPELINE DI INFERENZA ---
def run_pipeline(msg_path, conv_path):
    # Scarica il modello da MLflow
    model_weights_path = download_best_model()

    print(">>> Caricamento file CSV...")
    df_msg = pd.read_csv(msg_path, encoding='utf-8-sig', dtype=dtype_dict_1)
    df_conv_template = pd.read_csv(conv_path, encoding='utf-8-sig', dtype=dtype_dict_2)

    tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
    model = HierarchicalModel(MODEL_NAME).to(DEVICE)
    model.load_state_dict(torch.load(model_weights_path, map_location=DEVICE))
    model.eval()

    loader = DataLoader(InferenceDataset(df_msg, tokenizer), batch_size=1)

    results_msgs, results_convs = [], []

    print(">>> Classificazione in corso...")
    with torch.no_grad():
        for b in tqdm(loader):
            m_log, c_m_log, c_b_log = model(b['ids'].to(DEVICE), b['mask'].to(DEVICE), b['actual_len'].to(DEVICE))
            
            # Predizioni Messaggi
            m_preds = torch.argmax(m_log, dim=-1).squeeze(0).cpu().numpy()
            indices = b['indices']
            roles = b['roles']
            for i in range(b['actual_len'].item()):
                if roles[i][0] != 'assistant':
                    res = {'index': indices[i].item(), 'conversation_id': b['conv_id'][0]}
                    for h_idx, col in enumerate(MSG_LABEL_COLS): res[col] = m_preds[i, h_idx]
                    results_msgs.append(res)

            # Predizioni Conversazione
            c_m_preds = torch.argmax(c_m_log, dim=-1).squeeze(0).cpu().numpy()
            c_b_pred = torch.argmax(c_b_log, dim=-1).item()
            c_res = {'conversation_id': b['conv_id'][0], 'global_risk_flag': c_b_pred}
            for h_idx, col in enumerate(CONV_LABEL_COLS): c_res[col] = c_m_preds[h_idx]
            results_convs.append(c_res)

    # Salvataggio
    df_res_msgs = pd.DataFrame(results_msgs).set_index('index')
    for col in MSG_LABEL_COLS:
        df_msg.loc[df_res_msgs.index, col] = df_res_msgs[col]
    
    df_res_convs = pd.DataFrame(results_convs)
    df_conv_final = df_conv_template.drop(columns=CONV_LABEL_COLS + ['global_risk_flag'], errors='ignore')
    df_conv_final = pd.merge(df_conv_final, df_res_convs, on='conversation_id', how='left')

    df_msg.to_csv("nuovo_dataset_5000_CLASSIFIED.csv", index=False, encoding='utf-8-sig')
    df_conv_final.to_csv("nuovo_dataset_appoggio_CLASSIFIED.csv", index=False, encoding='utf-8-sig')
    print(">>> Processo completato!")


In [14]:
run_pipeline('/kaggle/input/aise-labeling-inferenza/nuovo_dataset_5000_ridotto_1.csv', '/kaggle/input/aise-labeling-inferenza/nuovo_dataset_appoggio_ridotto_1.csv')

>>> Ricerca dell'ultimo modello in corso nell'esperimento: Social_Risk_Multilingual_4...
>>> Download artefatto dalla Run ID: 1794b8a49dbe49b4b76ffb4ed32b3c54


>>> Modello scaricato correttamente in: /tmp/tmp7igaqq1g/best_model.pt
>>> Caricamento file CSV...


Preparazione dati: 100%|██████████| 30/30 [00:00<00:00, 4512.43it/s]


>>> Classificazione in corso...


100%|██████████| 30/30 [01:47<00:00,  3.58s/it]

>>> Processo completato!
